# Cox & competing-risks survival

**Survival analysis is a temporal point process.** A subject is followed until an
outcome fires once (the *terminal event*) or the observation window ends without
it (*right-censoring*). TorchPP models this directly, and two TPP quantities are
exactly the survival-analysis quantities:

- the **intensity** $\lambda(t\mid x)$ *is* the **hazard** $h(t\mid x)$ — the
  instantaneous event rate given covariates $x$;
- the **survival function** is the exponential of the negative cumulative
  intensity, $S(t) = \exp\!\big(-\Lambda(t)\big)$ with $\Lambda(t)=\int_0^t \lambda(u)\,du$.

We fit the **Cox proportional-hazards** model
$h(t\mid x) = h_0(t)\,\exp(x\cdot\beta)$. Here the baseline hazard $h_0$ is
*constant in time* (an exponential model); notebook 02 makes it time-varying.

**Two model flags carry the survival semantics:**

- `terminating=True` — each event type fires at most once. Its intensity drops
  to zero after it occurs, so the survival integral runs to the event time
  (uncensored) or to the window end `T_obs` (censored).
- `has_origin=True` — the single event *is* the scored outcome. (With
  `has_origin=False` the first event would be treated as a conditioned-on origin
  and dropped — wrong for survival.)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from torchpp import IntensityBasedTPP, TokenRoles
from survival_tools import (
    make_exponential_ph, make_competing_risks,
    survival_batch, covariate_batch, fit_full_batch,
    model_survival_curve, cumulative_incidence,
    kaplan_meier, aalen_johansen,
    plot_coefficients, plot_survival_km_vs_model, plot_cumulative_incidence,
)

# survival_tools selects the Agg backend on import; switch to inline afterwards
# so the figures render in the notebook.
%matplotlib inline
_ = torch.manual_seed(0)

## 1. Synthetic exponential proportional-hazards data

`make_exponential_ph` draws covariates $x\sim\mathcal N(0, I)$ and an event time
$T\sim\text{Exp}\big(h_0\,e^{x\cdot\beta}\big)$, then right-censors anything past
`T_obs`. A censored subject becomes an *empty* sequence that still carries its
covariates and window; an observed subject carries its single event.

In [ ]:
BETA_TRUE = torch.tensor([0.8, -0.5, 0.3])   # true log-hazard-ratio per feature
H0_TRUE, T_OBS, N = 0.5, 4.0, 3000

seqs, meta = make_exponential_ph(n=N, beta=BETA_TRUE, h0=H0_TRUE, T_obs=T_OBS, seed=0)
batch = survival_batch(seqs)

censoring = 1.0 - meta["observed"].float().mean()
print(f"n = {N}   censoring rate = {censoring:.1%}\n")

# one uncensored subject (a single terminal event) and one censored (empty history)
obs = meta["observed"]
i_obs = int(torch.nonzero(obs)[0])
i_cen = int(torch.nonzero(~obs)[0])
print("UNCENSORED subject:\n", seqs[i_obs], "\n")
print("CENSORED subject:\n", seqs[i_cen])

## 2. The Cox model as a custom TPP

The model is a short subclass of `IntensityBasedTPP`. TorchPP already stores and
batches the per-subject covariates under the key `"x"`; the intensity function
just reads them. We keep a learnable log baseline hazard and a coefficient vector.

We write it once for a general number of causes `D` — with `D=1` it is the ordinary
single-event Cox model; a later cell reuses the **same class** with `D>1` for
competing risks. Three hooks define the model:

- `_rate` — the per-subject, per-cause hazard $h_{0,d}\,e^{x\cdot\beta_d}$,
  constant in time;
- `_intensities_at_queries` — the required intensity primitive, here the rate
  broadcast across the query times;
- `_closed_form_ci` — the analytic cumulative intensity $\Lambda$. `_effective_T`
  returns the per-cause upper limit $\min(T, t_{\text{term}})$, which is where the
  terminating semantics enter.

In [ ]:
class CoxTPP(IntensityBasedTPP):
    '''Exponential proportional-hazards survival (D=1) or competing risks (D>1).

    Constant-in-time hazard per cause: h_d(t | x) = exp(log_h0[d]) * exp(x . beta[d]).
    '''

    def __init__(self, n_features, D=1):
        super().__init__(D=D, terminating=True, token_roles=TokenRoles(has_origin=True))
        self.log_h0 = nn.Parameter(torch.zeros(D))            # (D,) log baseline hazard
        self.beta = nn.Parameter(torch.zeros(D, n_features))  # (D, n_features) coefficients

    def _rate(self, ts):
        # Per-subject, per-cause hazard from the static covariates.  (B, D).
        x = ts.covariates["x"]                                # (B, n_features)
        return torch.exp(self.log_h0) * torch.exp(x @ self.beta.T)  # (B, D)

    def _intensities_at_queries(self, query_times, ts):
        # Constant-in-time hazard, broadcast over the Q query times.  (B, Q, D).
        B, Q = query_times.shape
        return self._rate(ts).view(B, 1, self.D).expand(B, Q, self.D)

    def _closed_form_ci(self, ts, T_low, T):
        # Lambda_d over [T_low or 0, T] = rate_d * (hi_d - lo_d).  _effective_T folds in
        # the terminating per-cause limit: the event time once a cause fires, else T.
        hi = self._effective_T(T, ts)                                    # (B, D)
        lo = self._effective_T(T_low, ts) if T_low is not None else 0.0  # (B, D) or 0
        return self._rate(ts) * (hi - lo)                                # (B, D)

## 3. Fit by maximum likelihood

`fit_full_batch` (from `survival_tools`) runs full-batch Adam on the per-event negative log-likelihood. Uncensored
subjects contribute a log-hazard term at their event time; every subject
contributes its survival integral over the window.

In [ ]:
model = CoxTPP(n_features=3, D=1)
history = fit_full_batch(model, batch, steps=800, lr=0.05, log_every=200)
print(f"\nfinal loss (NLL/event) = {history['losses'][-1]:.4f}")

## 4. Recover the coefficients

The learned $\beta$ should match the truth. Each coefficient is a **log hazard
ratio**: $e^{\beta_j}$ is the multiplicative change in hazard per unit increase in
feature $j$ (>1 raises risk, <1 lowers it).

In [ ]:
beta_hat = model.beta.detach().reshape(-1)
h0_hat = torch.exp(model.log_h0.detach()).item()
print("true beta      :", [round(float(b), 3) for b in BETA_TRUE])
print("recovered beta :", [round(float(b), 3) for b in beta_hat])
print("hazard ratios  :", [round(float(torch.exp(b)), 3) for b in beta_hat])
print(f"true h0 = {H0_TRUE}   recovered h0 = {h0_hat:.3f}")

fig = plot_coefficients(BETA_TRUE, beta_hat)
plt.show()

## 5. Predicted survival vs Kaplan-Meier, by risk stratum

Survival and hazard curves are evaluated on an empty-history batch built by
`covariate_batch` (see [Predictive queries](./README.md#predictive-queries)).

We split subjects into three strata by their true linear risk $x\cdot\beta$ and,
for each, compare the non-parametric **Kaplan-Meier** estimate (dashed) with the
model's survival curve at the stratum's mean covariate (solid).

In [ ]:
event_time = meta["event_time"]
obs_np = meta["observed"].numpy()
obs_time = torch.minimum(event_time, torch.tensor(T_OBS)).numpy()  # min(event, censor)

risk = (meta["x"] @ BETA_TRUE).numpy()
cuts = np.quantile(risk, [1/3, 2/3])
stratum = np.digitize(risk, cuts)              # 0 = low risk, 1 = mid, 2 = high

t_grid = np.linspace(0.0, T_OBS, 200)
km_t, km_S, S_model = [], [], []
for s in range(3):
    m = stratum == s
    tk, sk = kaplan_meier(obs_time[m], obs_np[m])
    km_t.append(tk); km_S.append(sk)
    x_rep = meta["x"][m].mean(dim=0, keepdim=True)          # stratum mean covariate
    S_model.append(model_survival_curve(model, x_rep, t_grid).numpy()[0])

fig = plot_survival_km_vs_model(
    km_t, km_S, t_grid, np.array(S_model),
    labels=["low risk", "mid risk", "high risk"],
)
plt.show()

## 6. Extending to competing risks

Often a subject can fail from one of several **mutually exclusive** causes (e.g.
death from disease A vs disease B): whichever fires first ends follow-up and
censors the others. This is a **multivariate** terminating TPP — one event type
per cause — and needs no new model. We reuse `CoxTPP` with `D>1`: now `log_h0`
holds one baseline per cause and `beta` one coefficient row per cause.

In [ ]:
BETAS_TRUE = torch.tensor([[0.9, -0.3],    # cause 0 coefficients
                           [-0.5, 0.6]])   # cause 1 coefficients
H0S_TRUE = torch.tensor([0.4, 0.6])        # per-cause baseline hazard
T_OBS_CR, N_CR = 5.0, 5000

cr_seqs, cr_meta = make_competing_risks(
    n=N_CR, betas=BETAS_TRUE, h0=H0S_TRUE, T_obs=T_OBS_CR, seed=1,
)
cr_batch = survival_batch(cr_seqs)
cr_cens = 1.0 - cr_meta["observed"].float().mean()
print(f"n = {N_CR}   censoring rate = {cr_cens:.1%}")
for d in range(2):
    frac = (cr_meta["cause"][cr_meta["observed"]] == d).float().mean()
    print(f"  cause {d}: {frac:.1%} of observed events")

In [ ]:
cr_model = CoxTPP(n_features=2, D=2)   # SAME class, D=2
cr_history = fit_full_batch(cr_model, cr_batch, steps=1000, lr=0.05, log_every=250)
print(f"\nfinal loss (NLL/event) = {cr_history['losses'][-1]:.4f}")

## 7. Recover per-cause coefficients

Each cause has its own $\beta_d$; a covariate can raise the hazard of one cause
while lowering another. All are recovered from the single fit.

In [ ]:
for d in range(2):
    bd_hat = cr_model.beta[d].detach()
    h0d = torch.exp(cr_model.log_h0[d].detach()).item()
    print(f"cause {d}: true beta {[round(float(b),3) for b in BETAS_TRUE[d]]}  "
          f"recovered {[round(float(b),3) for b in bd_hat]}  "
          f"| h0 {float(H0S_TRUE[d]):.2f} -> {h0d:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for d in range(2):
    plot_coefficients(BETAS_TRUE[d], cr_model.beta[d].detach(), ax=axes[d])
    axes[d].set_title(f"cause {d}")
plt.tight_layout()
plt.show()

## 8. Cumulative incidence

With competing risks, "the probability of failing from cause $d$ by time $t$" is
the **cumulative incidence function**
$\text{CIF}_d(t) = \int_0^t S(u)\,h_d(u)\,du$, where $S$ is the *all-cause*
survival. It accounts for the competition: a subject can only fail from cause $d$
if no cause has fired yet. We compare the model's CIF, averaged over the
covariates of all subjects, with the non-parametric **Aalen-Johansen** estimate of
the whole cohort. (A curve at one covariate value, e.g. $x=0$, is a different
quantity and need not match the cohort estimate.)

In [ ]:
cr_grid = np.linspace(0.0, T_OBS_CR, 200)
# per-subject CIF_d(t | x_i), averaged over the cohort -> (D, G)
cif_model = cumulative_incidence(cr_model, cr_meta["x"], cr_grid).mean(dim=0).numpy()

aj_time = np.minimum(cr_meta["event_time"].numpy(), T_OBS_CR)
aj_t, aj_cif = aalen_johansen(
    aj_time, cr_meta["cause"].numpy(), cr_meta["observed"].numpy(), n_causes=2,
)

print("model CIF at t_end :", [round(float(cif_model[d][-1]), 3) for d in range(2)])
print("A-J   CIF at t_end :", [round(float(aj_cif[d][-1]), 3) for d in range(2)])

fig = plot_cumulative_incidence(cr_grid, cif_model, aj_t, aj_cif)
plt.show()

## Takeaways

- A survival model **is** a terminating single-event TPP: the intensity is the
  hazard, and $S(t)=\exp(-\Lambda(t))$ falls out of the cumulative intensity.
- Covariate modulation is just an intensity that reads `ts.covariates` — no change
  to the library, only a short subclass.
- **Competing risks come for free** from the multivariate machinery. The mechanism
  is `_effective_T`: for a terminating process it caps each cause's integral at
  $\min(T, t_{\text{term}})$, so once any cause fires, every cause stops accruing
  hazard at that shared time — exactly the competing-risks likelihood.